In [ ]:
from typing import TypedDict


class SubGraphState(TypedDict):
    sub_value: str


def subgraph_node_1(state: SubGraphState) -> SubGraphState:
    return {"sub_value": f"你好 {state['sub_value']}"}


from langgraph.graph.state import END, START, StateGraph

sub_graph = (
    StateGraph(SubGraphState)
    .add_node("subgraph_node", subgraph_node_1)
    .add_edge(START, "subgraph_node")
    .add_edge("subgraph_node", END)
    .compile()
)


class MainGraphState(TypedDict):
    main_value: str


def call_subgraph_node(state: MainGraphState) -> MainGraphState:
    """调用子图节点

    该节点充当了适配器
    - 转换子图的输入
    - 转换子图的输出
    """
    subgraph_output: SubGraphState = sub_graph.invoke({"sub_value": state["main_value"]})
    return {"main_value": subgraph_output["sub_value"]}


main_graph = (
    StateGraph(MainGraphState)
    .add_node("call_subgraph_node", call_subgraph_node)
    .add_edge(START, "call_subgraph_node")
    .add_edge("call_subgraph_node", END)
    .compile()
)

async for chunk in main_graph.astream({"main_value": "今天天气不错"}, subgraphs=True, stream_mode="debug"):
    print(chunk)


((), {'step': 1, 'timestamp': '2026-07-08T03:47:21.886973+00:00', 'type': 'task', 'payload': {'id': 'f6231929-fedf-3900-b259-1df29ed236d3', 'name': 'call_subgraph_node', 'input': {'main_value': '今天天气不错'}, 'triggers': ('branch:to:call_subgraph_node',)}})
(('call_subgraph_node:f6231929-fedf-3900-b259-1df29ed236d3',), {'step': 1, 'timestamp': '2026-07-08T03:47:21.888425+00:00', 'type': 'task', 'payload': {'id': '0efb2da4-a428-7a1b-35ce-22d996437fe1', 'name': 'subgraph_node', 'input': {'sub_value': '今天天气不错'}, 'triggers': ('branch:to:subgraph_node',)}})
(('call_subgraph_node:f6231929-fedf-3900-b259-1df29ed236d3',), {'step': 1, 'timestamp': '2026-07-08T03:47:21.888613+00:00', 'type': 'task_result', 'payload': {'id': '0efb2da4-a428-7a1b-35ce-22d996437fe1', 'name': 'subgraph_node', 'error': None, 'result': {'sub_value': '你好 今天天气不错'}, 'interrupts': []}})
((), {'step': 1, 'timestamp': '2026-07-08T03:47:21.889261+00:00', 'type': 'task_result', 'payload': {'id': 'f6231929-fedf-3900-b259-1df29ed236